# 🌳 Clase 6 — Árboles de Decisión y Random Forest (R)

### Aplicaciones de Minería de Datos a Economía y Finanzas
**Maestría en Minería de Datos · UTN Facultad Regional Rosario · 2026**

*Dr. Darío Ezequiel Díaz · Jueves 14 de mayo de 2026*

---

## 📋 Objetivos de la práctica en R

Al finalizar este notebook, ustedes serán capaces de:

1. **Entrenar** un árbol de decisión CART sobre el German Credit Dataset usando el paquete `rpart` (Therneau & Atkinson), con visualización mediante `rpart.plot`.
2. **Diagnosticar** el sobreajuste arbóreo trazando las curvas de error de entrenamiento y validación en función de la profundidad máxima.
3. **Aplicar** post-poda por *cost-complexity pruning* utilizando el parámetro `cp` y el método `printcp()` propio de rpart.
4. **Construir** un Random Forest con el paquete clásico `randomForest` (Liaw & Wiener, port del código Fortran original de Breiman & Cutler).
5. **Computar** la importancia de variables por dos enfoques nativos del paquete: *Mean Decrease in Gini* (análogo al MDI de sklearn) y *Mean Decrease in Accuracy* (análogo al permutation importance).
6. **Comparar** el rendimiento de CART y Random Forest contra la regresión logística de la Clase 5, mediante AUC, KS y Gini.

## 🗺️ Mapa del notebook

| Sección | Contenido |
|---------|-----------|
| 1 | Configuración del entorno R en Colab |
| 2 | Carga del dataset German Credit |
| 3 | Preprocesamiento y partición train/test |
| 4 | Árbol de decisión CART con `rpart` |
| 5 | Análisis del sobreajuste y poda |
| 6 | Random Forest con `randomForest` |
| 7 | Importancia de variables: MDG vs MDA |
| 8 | Comparativa final con regresión logística |
| 9 | Síntesis y entregable |

---

## 🔄 Diferencias filosóficas R vs Python

| Aspecto | Python (sklearn) | R (`rpart` + `randomForest`) |
|---------|------------------|------------------------------|
| Encoding de categóricas | **Requiere one-hot manual** | **Factores nativos** sin transformación |
| Parámetro de poda | `ccp_alpha` | `cp` (complexity parameter) |
| Selección automática de cp | Manual con CV | **`printcp()` y `plotcp()` integradas** |
| Visualización del árbol | `plot_tree()` (texto recortado) | **`rpart.plot()` (profesional)** |
| Random Forest API | `RandomForestClassifier` | `randomForest()` (port del original Fortran) |
| Hiperparámetro nº árboles | `n_estimators` | `ntree` |
| Hiperparámetro features/split | `max_features` | `mtry` |
| Importancia nativa | MDI (impurezas) | **MDG (Gini) + MDA (accuracy)** |
| Permutation importance | `sklearn.inspection` | **Incluida en `randomForest`** |

> 💡 R es históricamente el lenguaje de referencia para árboles y bosques en banca regulada: los paquetes `rpart` y `randomForest` son ports directos de la implementación original de Breiman y se mantienen como estándar industrial desde hace décadas.

---

> 📌 **Recomendación:** ejecuten las celdas en orden secuencial. La sección 1 carga el setup script personalizado del curso con ~237 paquetes pre-instalados en su carpeta de Drive.


## 1. Configuración del entorno R en Colab

Cargamos el setup script personal del curso ubicado en `/content/drive/MyDrive/R_Colab/setup_R_colab.R`. Este script tiene aproximadamente 237 paquetes pre-instalados, incluyendo `rpart`, `rpart.plot`, `randomForest`, `caret` y todo el ecosistema `tidyverse`.

### Paquetes específicos de Clase 6

- **`rpart`** — Therneau & Atkinson; implementación canónica de CART en R
- **`rpart.plot`** — Milborrow; visualización profesional de árboles
- **`randomForest`** — Liaw & Wiener; port directo del código Fortran de Breiman & Cutler
- **`pROC`** — curvas ROC y AUC
- **`vip`** — interpretación de variables (opcional, alternativa moderna)
- **`tidyverse`** — manipulación y visualización

In [ ]:
# Cargar setup script del curso (ruta personal en Drive)
# Si no está disponible, descomentar el bloque de instalación manual abajo

ruta_setup <- "/content/drive/MyDrive/R_Colab/setup_R_colab.R"

if (file.exists(ruta_setup)) {
  source(ruta_setup)
  cat("✅ Setup script cargado desde Drive\n")
  cat("   ~237 paquetes pre-instalados disponibles\n")
} else {
  cat("⚠️  Setup script no disponible. Instalación manual:\n\n")
  cat("install.packages(c('tidyverse', 'rpart', 'rpart.plot',\n")
  cat("                   'randomForest', 'pROC', 'scorecard'),\n")
  cat("                 repos = 'https://cran.rstudio.com/',\n")
  cat("                 dependencies = TRUE)\n")
}

In [ ]:
# Algunos paquetes pueden faltar en el setup; los instalamos puntualmente
paquetes_clase6 <- c("rpart", "rpart.plot", "randomForest", "pROC", "scorecard")

for (pkg in paquetes_clase6) {
  if (!requireNamespace(pkg, quietly = TRUE)) {
    cat(sprintf("📦 Instalando %s...\n", pkg))
    install.packages(pkg, repos = "https://cran.rstudio.com/", quiet = TRUE)
  }
}
cat("\n✅ Todos los paquetes de Clase 6 disponibles\n")

In [ ]:
# Carga de librerías
suppressPackageStartupMessages({
  library(tidyverse)      # manipulación + ggplot2
  library(rpart)          # CART
  library(rpart.plot)     # visualización del árbol
  library(randomForest)   # bosque aleatorio
  library(pROC)           # curvas ROC y AUC
  library(scorecard)      # dataset germancredit interno
})

# Configuración global del curso
RNG_SEED <- 2026
set.seed(RNG_SEED)

# ═══ Paleta cromática Unidad 3: Royal Indigo & Amber ═══
COLOR_INDIGO   <- "#1A2148"
COLOR_VIOLET   <- "#6E3FD0"
COLOR_AMBER    <- "#D97706"
COLOR_EMERALD  <- "#14B8A6"
COLOR_CARMIN   <- "#DC2626"
COLOR_GOLD     <- "#F5C443"
COLOR_GRAY     <- "#64748B"

# Tema ggplot2 personalizado
tema_unidad3 <- theme_minimal(base_size = 11) +
  theme(
    plot.title = element_text(size = 13, face = "bold", color = COLOR_INDIGO),
    plot.subtitle = element_text(size = 10, color = COLOR_GRAY),
    axis.title = element_text(face = "bold"),
    panel.grid.minor = element_blank(),
    legend.position = "right"
  )

theme_set(tema_unidad3)

cat(sprintf("🌳 RNG_SEED = %d\n", RNG_SEED))
cat("🎨 Paleta cromática Unidad 3 cargada\n")
cat("📚 Librerías importadas: tidyverse, rpart, rpart.plot, randomForest, pROC, scorecard\n")

## 2. Carga del German Credit Dataset

El paquete `scorecard` incluye el German Credit Dataset listo para usar mediante `data(germancredit)`. Esta es la fuente preferida en R: las variables vienen tipificadas correctamente como factores con sus niveles ordenados, sin necesidad de descargas externas.

**Variables**: 20 covariables (13 categóricas como factor, 7 numéricas) y la variable respuesta `creditability` con niveles `'good'` (700 casos, 70 %) y `'bad'` (300 casos, 30 %).

Recodificamos siguiendo la convención regulatoria: **`Y = 1` representa default**, `Y = 0` representa pago regular.

In [ ]:
# Función de carga resiliente
cargar_german_credit <- function() {

  # Intento 1: dataset interno del paquete scorecard
  resultado <- try({
    data(germancredit, package = "scorecard", envir = environment())
    germancredit
  }, silent = TRUE)

  if (!inherits(resultado, "try-error")) {
    cat(sprintf("✅ Dataset cargado desde scorecard::germancredit: %d × %d\n",
                nrow(resultado), ncol(resultado)))
    return(resultado)
  }

  # Intento 2: URL pública de OpenML
  url_openml <- "https://www.openml.org/data/get_csv/31/dataset_31_credit-g.arff"
  resultado <- try(read.csv(url_openml), silent = TRUE)
  if (!inherits(resultado, "try-error")) {
    cat(sprintf("✅ Dataset cargado desde URL pública: %d × %d\n",
                nrow(resultado), ncol(resultado)))
    return(resultado)
  }

  # Intento 3: Drive personal
  ruta_drive <- "/content/drive/MyDrive/datasets/german_credit.csv"
  resultado <- try(read.csv(ruta_drive), silent = TRUE)
  if (!inherits(resultado, "try-error")) {
    cat(sprintf("✅ Dataset cargado desde Drive: %d × %d\n",
                nrow(resultado), ncol(resultado)))
    return(resultado)
  }

  stop("❌ No se pudo cargar el dataset por ninguna vía.")
}

df <- cargar_german_credit()
head(df)

In [ ]:
# Recodificación de la variable respuesta según convención regulatoria
# Y = 1 → default (clase positiva, evento de interés)
# Y = 0 → pago regular
#
# El dataset scorecard::germancredit trae 'creditability' como factor con niveles 'good'/'bad'

df$Y <- factor(ifelse(df$creditability == "bad", 1, 0), levels = c(0, 1))
df$creditability <- NULL  # eliminamos la versión textual

cat("📊 Distribución de la variable respuesta:\n\n")
tabla_Y <- table(Y = df$Y)
print(tabla_Y)

prop_default <- mean(df$Y == 1)
cat(sprintf("\n   Y = 0 (pago regular): %4d casos (%.1f%%)\n",
            sum(df$Y == 0), (1 - prop_default) * 100))
cat(sprintf("   Y = 1 (default):      %4d casos (%.1f%%)\n",
            sum(df$Y == 1), prop_default * 100))
cat(sprintf("\n   Ratio buenos:malos = %.2f:1\n", sum(df$Y == 0) / sum(df$Y == 1)))

## 3. Preprocesamiento y partición

### 🌟 Ventaja arquitectónica de R

A diferencia de scikit-learn en Python, los paquetes `rpart` y `randomForest` **trabajan nativamente con variables tipo `factor`**. Esto significa que:

- ❌ **No necesitamos one-hot encoding** — los modelos manejan internamente las particiones binarias óptimas sobre los niveles categóricos.
- ❌ **No necesitamos estandarizar variables numéricas** — los árboles son invariantes a transformaciones monótonas.
- ✅ **El modelo conserva los nombres originales de las variables**, lo cual mejora la interpretabilidad de los splits y de la importancia de variables.

El único preprocesamiento necesario es la **partición train/test estratificada** preservando la prevalencia del default.

In [ ]:
# Inspección de tipos de variables
cat("📦 Estructura del dataset:\n\n")
tipos <- sapply(df, class)
n_factor <- sum(tipos %in% c("factor", "ordered"))
n_numeric <- sum(tipos %in% c("numeric", "integer"))

cat(sprintf("   Variables factor:    %d\n", n_factor))
cat(sprintf("   Variables numéricas: %d\n", n_numeric - 1))  # -1 por Y
cat(sprintf("   Variable objetivo:   Y (factor binario)\n\n"))

# Listar variables por tipo
vars_factor <- names(df)[sapply(df, function(x) inherits(x, c("factor","ordered"))) & names(df) != "Y"]
vars_numeric <- names(df)[sapply(df, function(x) is.numeric(x))]

cat("   Factores: ", paste(vars_factor[1:5], collapse = ", "), "...\n", sep = "")
cat("   Numéricas:", paste(vars_numeric, collapse = ", "), "\n", sep = " ")

In [ ]:
# Partición train/test estratificada por Y
# Usamos createDataPartition de caret si está disponible, de lo contrario muestreo manual

set.seed(RNG_SEED)

# Muestreo estratificado manual: 80% de cada clase a train
idx_train_0 <- sample(which(df$Y == 0), size = round(0.8 * sum(df$Y == 0)))
idx_train_1 <- sample(which(df$Y == 1), size = round(0.8 * sum(df$Y == 1)))
idx_train <- c(idx_train_0, idx_train_1)

train <- df[idx_train, ]
test  <- df[-idx_train, ]

cat("📊 Partición train/test estratificada:\n")
cat(sprintf("   Train: %d obs × %d features  (default = %.1f%%)\n",
            nrow(train), ncol(train) - 1, mean(train$Y == 1) * 100))
cat(sprintf("   Test:  %d obs × %d features  (default = %.1f%%)\n",
            nrow(test), ncol(test) - 1, mean(test$Y == 1) * 100))
cat("\n✅ Proporción de default preservada en ambos splits\n")

## 4. Árbol de decisión CART con `rpart`

El paquete `rpart` es la implementación canónica de CART en R. Ajustamos un primer árbol con hiperparámetros razonables:

- `method = "class"` — clasificación (alternativas: `"anova"` para regresión).
- `parms = list(split = "gini")` — criterio de impureza Gini (default; alternativa: `"information"` para entropía).
- `control = rpart.control(...)` — hiperparámetros del crecimiento del árbol.

### Equivalencias de hiperparámetros R vs Python

| Python (sklearn) | R (rpart) | Función |
|------------------|-----------|---------|
| `max_depth` | `maxdepth` | Profundidad máxima |
| `min_samples_split` | `minsplit` | Mínimo de obs. para considerar partición |
| `min_samples_leaf` | `minbucket` | Mínimo de obs. por hoja terminal |
| `ccp_alpha` | `cp` | Parámetro de complejidad |

> 📌 Importante: el parámetro `cp` de rpart cumple un rol levemente distinto al `ccp_alpha` de sklearn. En rpart, `cp` controla el crecimiento durante la construcción (pre-poda), pero el árbol completo contiene su propia secuencia interna de poda accesible vía `printcp()`.

In [ ]:
# Entrenamiento del árbol CART inicial
set.seed(RNG_SEED)

cart_inicial <- rpart(
  Y ~ .,                                # fórmula: Y en función de todas las demás
  data    = train,
  method  = "class",                    # clasificación
  parms   = list(split = "gini"),       # criterio Gini
  control = rpart.control(
    maxdepth  = 5,                      # profundidad máxima
    minsplit  = 40,                     # mínimo de obs para considerar split
    minbucket = 20,                     # mínimo de obs por hoja
    cp        = 0.001                   # crecimiento permisivo (poda fina después)
  )
)

# Predicciones de probabilidad sobre train y test
p_train_cart_ini <- predict(cart_inicial, train, type = "prob")[, "1"]
p_test_cart_ini  <- predict(cart_inicial, test,  type = "prob")[, "1"]

# AUC sobre ambos conjuntos
auc_train_cart_ini <- as.numeric(auc(train$Y, p_train_cart_ini, quiet = TRUE))
auc_test_cart_ini  <- as.numeric(auc(test$Y,  p_test_cart_ini,  quiet = TRUE))

cat("🌳 Árbol CART inicial (maxdepth=5, minbucket=20, cp=0.001)\n\n")
cat(sprintf("   Número de nodos:   %d\n", nrow(cart_inicial$frame)))
cat(sprintf("   Número de hojas:   %d\n", sum(cart_inicial$frame$var == "<leaf>")))
cat("\n📈 Performance:\n")
cat(sprintf("   AUC train: %.4f\n", auc_train_cart_ini))
cat(sprintf("   AUC test:  %.4f\n", auc_test_cart_ini))
cat(sprintf("   Brecha:    %.4f  (indicador de sobreajuste)\n",
            auc_train_cart_ini - auc_test_cart_ini))

### 4.1. Visualización del árbol con `rpart.plot`

`rpart.plot` produce diagramas significativamente más legibles que `plot_tree()` de sklearn. Cada nodo muestra la regla de partición, la probabilidad estimada de cada clase, y el porcentaje de observaciones que llegan al nodo.

Parámetros usados:
- `type = 4` — estilo extendido con etiquetas en cada rama
- `extra = 104` — muestra probabilidad y porcentaje de observaciones
- `box.palette = c("Greens", "Reds")` — degradado del verde (OK) al rojo (default)

In [ ]:
# Visualización del árbol con rpart.plot
options(repr.plot.width = 14, repr.plot.height = 8)

rpart.plot(
  cart_inicial,
  type        = 4,                              # estilo extendido
  extra       = 104,                            # probabilidad + % obs
  box.palette = c("Greens", "Reds"),            # OK verde, default rojo
  branch.lty  = 1,                              # línea sólida en ramas
  nn          = FALSE,                          # ocultar números de nodos
  fallen.leaves = TRUE,                         # hojas alineadas al fondo
  main        = "Árbol CART — German Credit (maxdepth=5)",
  cex.main    = 1.1
)

### 4.2. Tabla de complejidad: `printcp()`

Una característica distintiva de `rpart` es que la función calcula internamente, durante el entrenamiento, una **tabla de complejidad** que muestra el error de validación cruzada para cada posible valor de poda. Esta tabla se accede con `printcp()` y es la base del análisis de sobreajuste.

In [ ]:
# Tabla de complejidad: error CV interna del árbol
cat("📊 Tabla de complejidad de rpart (CV interna):\n\n")
printcp(cart_inicial)

cat("\n📖 Lectura de la tabla:\n")
cat("   CP        - Parámetro de complejidad\n")
cat("   nsplit    - Número de splits realizados\n")
cat("   rel error - Error relativo sobre entrenamiento\n")
cat("   xerror    - Error relativo por CV (Cross-Validation)\n")
cat("   xstd      - Desvío estándar del error CV\n")

## 5. Análisis del sobreajuste y poda

### 5.1. Curva de sobreajuste por profundidad

Replicamos el experimento canónico del notebook Python: entrenar árboles con profundidades crecientes y trazar AUC train vs AUC test. La **curva en U** del error de validación es el diagnóstico visual del sobreajuste.

In [ ]:
# Curva de sobreajuste: AUC train vs AUC test en función de maxdepth
profundidades <- 1:15
resultados_curva <- data.frame(
  profundidad = integer(),
  auc_train   = numeric(),
  auc_test    = numeric(),
  n_hojas     = integer()
)

for (d in profundidades) {
  set.seed(RNG_SEED)
  m <- rpart(
    Y ~ ., data = train, method = "class",
    control = rpart.control(maxdepth = d, minsplit = 2, minbucket = 1, cp = 0)
  )
  p_train <- predict(m, train, type = "prob")[, "1"]
  p_test  <- predict(m, test,  type = "prob")[, "1"]
  auc_tr <- as.numeric(auc(train$Y, p_train, quiet = TRUE))
  auc_ts <- as.numeric(auc(test$Y,  p_test,  quiet = TRUE))
  n_hojas <- sum(m$frame$var == "<leaf>")
  resultados_curva <- rbind(resultados_curva, data.frame(
    profundidad = d, auc_train = auc_tr, auc_test = auc_ts, n_hojas = n_hojas
  ))
}

resultados_curva$gap <- resultados_curva$auc_train - resultados_curva$auc_test
print(resultados_curva)

In [ ]:
# Visualización de la curva de sobreajuste
options(repr.plot.width = 10, repr.plot.height = 5.5)

idx_optimo <- which.max(resultados_curva$auc_test)
prof_optima <- resultados_curva$profundidad[idx_optimo]
auc_optimo  <- resultados_curva$auc_test[idx_optimo]

datos_largo <- resultados_curva %>%
  select(profundidad, auc_train, auc_test) %>%
  pivot_longer(cols = c(auc_train, auc_test),
               names_to = "conjunto", values_to = "auc") %>%
  mutate(conjunto = factor(conjunto,
                           levels = c("auc_train", "auc_test"),
                           labels = c("AUC entrenamiento", "AUC validación (test)")))

ggplot(datos_largo, aes(x = profundidad, y = auc, color = conjunto)) +
  geom_line(linewidth = 1.1) +
  geom_point(size = 2.6) +
  geom_vline(xintercept = prof_optima, linetype = "dashed",
             color = COLOR_CARMIN, linewidth = 0.8, alpha = 0.7) +
  annotate("point", x = prof_optima, y = auc_optimo,
           size = 6, color = COLOR_CARMIN) +
  annotate("text", x = prof_optima + 0.3, y = auc_optimo + 0.02,
           label = sprintf("Óptimo: depth=%d\nAUC=%.3f", prof_optima, auc_optimo),
           hjust = 0, color = COLOR_CARMIN, size = 3.5, fontface = "bold") +
  scale_color_manual(values = c("AUC entrenamiento" = COLOR_VIOLET,
                                 "AUC validación (test)" = COLOR_AMBER)) +
  scale_x_continuous(breaks = profundidades) +
  labs(
    title    = "Diagnóstico de sobreajuste: AUC train vs test por profundidad",
    subtitle = "Curva en U del error de validación",
    x = "Profundidad máxima del árbol",
    y = "AUC-ROC",
    color = NULL
  ) +
  theme(legend.position = "bottom")

cat(sprintf("\n🎯 Profundidad óptima por AUC test: %d (AUC=%.4f)\n",
            prof_optima, auc_optimo))
cat(sprintf("   Brecha train-test en óptimo: %.4f\n",
            resultados_curva$gap[idx_optimo]))

### 5.2. Cost-complexity pruning con `cp`

La estrategia más rigurosa de poda en `rpart` consiste en:

1. **Entrenar el árbol completo** con un `cp` muy bajo (p. ej.\ 0).
2. **Inspeccionar la tabla de complejidad** con `printcp()`.
3. **Identificar el `cp` óptimo** según la regla de Breiman: el mayor `cp` cuyo `xerror` quede dentro de una desviación estándar del mínimo (regla **1-SE**).
4. **Podar el árbol** con `prune()` usando ese `cp` óptimo.

Esta es la implementación canónica del **cost-complexity pruning de Breiman et al. (1984)** en R.

In [ ]:
# Árbol completo sin restricciones (cp = 0)
set.seed(RNG_SEED)
cart_pleno <- rpart(
  Y ~ ., data = train, method = "class",
  parms   = list(split = "gini"),
  control = rpart.control(cp = 0, minsplit = 2, minbucket = 1)
)

n_nodos_pleno <- nrow(cart_pleno$frame)
n_hojas_pleno <- sum(cart_pleno$frame$var == "<leaf>")

auc_train_pleno <- as.numeric(auc(train$Y, predict(cart_pleno, train, type="prob")[,"1"], quiet=TRUE))
auc_test_pleno  <- as.numeric(auc(test$Y,  predict(cart_pleno, test,  type="prob")[,"1"], quiet=TRUE))

cat("🌲 Árbol pleno (cp = 0, sin restricciones):\n")
cat(sprintf("   Nodos:       %d\n", n_nodos_pleno))
cat(sprintf("   Hojas:       %d\n", n_hojas_pleno))
cat(sprintf("   AUC train:   %.4f\n", auc_train_pleno))
cat(sprintf("   AUC test:    %.4f\n", auc_test_pleno))
cat(sprintf("   Brecha:      %.4f  (sobreajuste flagrante)\n",
            auc_train_pleno - auc_test_pleno))

In [ ]:
# Selección del cp óptimo por regla 1-SE
tabla_cp <- as.data.frame(cart_pleno$cptable)

# Mínimo xerror
idx_min <- which.min(tabla_cp$xerror)
xerror_min <- tabla_cp$xerror[idx_min]
xstd_min   <- tabla_cp$xstd[idx_min]

# Regla 1-SE: el menor árbol cuyo xerror queda dentro de 1 SD del mínimo
cp_1se_threshold <- xerror_min + xstd_min
idx_1se <- which(tabla_cp$xerror <= cp_1se_threshold)[1]  # el primer árbol que cumple

cp_optimo_min <- tabla_cp$CP[idx_min]
cp_optimo_1se <- tabla_cp$CP[idx_1se]

cat(sprintf("📊 Selección de cp por dos criterios:\n\n"))
cat(sprintf("   Criterio MÍNIMO:      cp = %.5f (xerror = %.4f)\n",
            cp_optimo_min, xerror_min))
cat(sprintf("   Criterio 1-SE:        cp = %.5f (xerror ≤ %.4f)\n",
            cp_optimo_1se, cp_1se_threshold))

cat("\n🎯 Usaremos la regla 1-SE: prefiere árboles más simples sin sacrificar significativamente performance.\n")

In [ ]:
# Visualización del proceso de selección con plotcp()
options(repr.plot.width = 10, repr.plot.height = 5.5)

plotcp(cart_pleno,
       col = COLOR_VIOLET,
       main = "Curva de cross-validation interna de rpart")
abline(h = cp_1se_threshold, lty = "dashed", col = COLOR_CARMIN, lwd = 2)

### 5.3. Árbol final podado

Aplicamos `prune()` con el `cp` óptimo seleccionado por la regla 1-SE. Comparamos contra el árbol inicial de la sección 4.

In [ ]:
# Árbol final podado con cp 1-SE
cart_final <- prune(cart_pleno, cp = cp_optimo_1se)

p_train_cart <- predict(cart_final, train, type = "prob")[, "1"]
p_test_cart  <- predict(cart_final, test,  type = "prob")[, "1"]

auc_train_cart <- as.numeric(auc(train$Y, p_train_cart, quiet = TRUE))
auc_test_cart  <- as.numeric(auc(test$Y,  p_test_cart,  quiet = TRUE))

cat(sprintf("🌳 CART final podado (cp = %.5f, regla 1-SE)\n", cp_optimo_1se))
cat(sprintf("   Nodos:        %d\n", nrow(cart_final$frame)))
cat(sprintf("   Hojas:        %d\n", sum(cart_final$frame$var == "<leaf>")))
cat("\n📈 Performance:\n")
cat(sprintf("   AUC train:    %.4f\n", auc_train_cart))
cat(sprintf("   AUC test:     %.4f\n", auc_test_cart))
cat(sprintf("   Brecha:       %.4f\n", auc_train_cart - auc_test_cart))

cat("\n📊 Comparación con árbol inicial:\n")
cat(sprintf("   Árbol inicial:  %d hojas, AUC test = %.4f, brecha = %.4f\n",
            sum(cart_inicial$frame$var == "<leaf>"), auc_test_cart_ini,
            auc_train_cart_ini - auc_test_cart_ini))
cat(sprintf("   Árbol final:    %d hojas, AUC test = %.4f, brecha = %.4f\n",
            sum(cart_final$frame$var == "<leaf>"), auc_test_cart,
            auc_train_cart - auc_test_cart))

In [ ]:
# Visualización del árbol final podado
options(repr.plot.width = 12, repr.plot.height = 7)

rpart.plot(
  cart_final,
  type        = 4,
  extra       = 104,
  box.palette = c("Greens", "Reds"),
  branch.lty  = 1,
  fallen.leaves = TRUE,
  main        = sprintf("Árbol CART final (cp = %.4f, regla 1-SE)", cp_optimo_1se),
  cex.main    = 1.1
)

## 6. Random Forest con `randomForest`

El paquete `randomForest` de Liaw y Wiener es el port directo del código Fortran original que Breiman y Cutler publicaron junto al paper seminal de 2001. Cuatro décadas después, sigue siendo la implementación de referencia en R.

### Equivalencias R vs Python

| sklearn | randomForest |
|---------|--------------|
| `n_estimators` | `ntree` |
| `max_features` | `mtry` |
| `min_samples_leaf` | `nodesize` |
| `class_weight` | `classwt` (peso por clase) |
| `oob_score=True` | OOB se calcula **siempre** automáticamente |
| `feature_importances_` | `importance(rf)` |

### Cálculo automático de `mtry`

`randomForest` selecciona `mtry = floor(sqrt(p))` por default en clasificación, idéntico al `max_features='sqrt'` de sklearn.

In [ ]:
# Random Forest con hiperparámetros recomendados
set.seed(RNG_SEED)

# mtry óptimo para clasificación = floor(sqrt(n_features))
n_features <- ncol(train) - 1  # excluyendo Y
mtry_default <- floor(sqrt(n_features))

rf <- randomForest(
  Y ~ .,
  data       = train,
  ntree      = 300,                          # número de árboles
  mtry       = mtry_default,                 # variables sorteadas por split
  nodesize   = 5,                            # mínimo obs por hoja
  classwt    = c("0" = 1, "1" = 2.33),       # peso inversamente proporcional
  importance = TRUE                          # habilita MDA + MDG
)

p_train_rf <- predict(rf, train, type = "prob")[, "1"]
p_test_rf  <- predict(rf, test,  type = "prob")[, "1"]

auc_train_rf <- as.numeric(auc(train$Y, p_train_rf, quiet = TRUE))
auc_test_rf  <- as.numeric(auc(test$Y,  p_test_rf,  quiet = TRUE))

# OOB error: el randomForest lo reporta como tasa de mal clasificación
# Calculamos accuracy OOB
oob_acc <- 1 - rf$err.rate[rf$ntree, "OOB"]

cat("🌲 Random Forest (ntree=300, mtry=", mtry_default, ")\n\n", sep="")
cat(sprintf("   Número de árboles:     %d\n", rf$ntree))
cat(sprintf("   Variables por split:   %d (≈ √%d)\n", mtry_default, n_features))
cat("\n📈 Performance:\n")
cat(sprintf("   AUC train:             %.4f\n", auc_train_rf))
cat(sprintf("   AUC test:              %.4f\n", auc_test_rf))
cat(sprintf("   Accuracy OOB:          %.4f\n", oob_acc))
cat("\n   👉 OOB se calcula automáticamente como subproducto del entrenamiento.\n")

### 6.1. Convergencia del error OOB

`randomForest` registra automáticamente el error OOB acumulado a medida que se agregan árboles. El atributo `rf$err.rate` contiene esa secuencia, lo cual permite graficar la convergencia sin reentrenar.

In [ ]:
# Visualización de la convergencia OOB
options(repr.plot.width = 10, repr.plot.height = 5.5)

err_df <- data.frame(
  n_trees     = 1:rf$ntree,
  oob_error   = rf$err.rate[, "OOB"],
  oob_clase_0 = rf$err.rate[, "0"],
  oob_clase_1 = rf$err.rate[, "1"]
) %>%
  pivot_longer(cols = starts_with("oob"),
               names_to = "tipo", values_to = "error") %>%
  mutate(tipo = factor(tipo,
                       levels = c("oob_error", "oob_clase_0", "oob_clase_1"),
                       labels = c("OOB global", "Error clase 0 (OK)", "Error clase 1 (Default)")))

ggplot(err_df, aes(x = n_trees, y = error, color = tipo)) +
  geom_line(linewidth = 1) +
  scale_color_manual(values = c("OOB global"             = COLOR_INDIGO,
                                "Error clase 0 (OK)"     = COLOR_EMERALD,
                                "Error clase 1 (Default)" = COLOR_CARMIN)) +
  labs(
    title    = "Convergencia del Random Forest: error OOB vs número de árboles",
    subtitle = sprintf("Estabilización a partir de ~100 árboles. ntree elegido = %d", rf$ntree),
    x = "Número de árboles",
    y = "Tasa de error",
    color = NULL
  ) +
  theme(legend.position = "bottom")

## 7. Importancia de variables: MDG vs MDA

`randomForest` con `importance = TRUE` calcula **dos medidas nativas** que corresponden a los enfoques que vimos en la teoría:

| randomForest | sklearn | Concepto |
|--------------|---------|----------|
| **MeanDecreaseGini (MDG)** | MDI | Reducción de impureza acumulada |
| **MeanDecreaseAccuracy (MDA)** | Permutation Importance | Caída de accuracy al permutar |

La diferencia respecto de Python es operativa, no conceptual: en R **ambas medidas vienen incluidas** en el resultado del entrenamiento, sin necesidad de invocar funciones adicionales como `permutation_importance()`.

### Sesgos conocidos

- **MDG** sesga hacia variables continuas y categóricas con alta cardinalidad (Strobl et al.\ 2007).
- **MDA** sigue siendo sensible a correlaciones entre predictoras, pero es insesgada respecto de cardinalidad.

In [ ]:
# Extraer ambas medidas de importancia
imp <- importance(rf)
imp_df <- data.frame(
  variable = rownames(imp),
  MDA      = imp[, "MeanDecreaseAccuracy"],
  MDG      = imp[, "MeanDecreaseGini"]
) %>%
  arrange(desc(MDG))

cat("📊 Top 15 variables por MDG (Mean Decrease in Gini):\n\n")
print(head(imp_df %>% arrange(desc(MDG)), 15))

cat("\n📊 Top 15 variables por MDA (Mean Decrease in Accuracy):\n\n")
print(head(imp_df %>% arrange(desc(MDA)), 15))

In [ ]:
# Visualización lado a lado: MDG vs MDA
options(repr.plot.width = 14, repr.plot.height = 7)

top_n <- 12

mdg_top <- imp_df %>% arrange(desc(MDG)) %>% head(top_n) %>%
  mutate(variable = factor(variable, levels = rev(variable)))

mda_top <- imp_df %>% arrange(desc(MDA)) %>% head(top_n) %>%
  mutate(variable = factor(variable, levels = rev(variable)))

p1 <- ggplot(mdg_top, aes(x = MDG, y = variable)) +
  geom_col(fill = COLOR_EMERALD, color = COLOR_INDIGO, linewidth = 0.3) +
  labs(title = "Top 12 — MDG (Mean Decrease in Gini)",
       x = "MDG", y = NULL)

p2 <- ggplot(mda_top, aes(x = MDA, y = variable)) +
  geom_col(fill = COLOR_AMBER, color = COLOR_INDIGO, linewidth = 0.3) +
  labs(title = "Top 12 — MDA (Mean Decrease in Accuracy)",
       x = "MDA", y = NULL)

# Usar gridExtra si está disponible, sino imprimir secuencialmente
if (requireNamespace("gridExtra", quietly = TRUE)) {
  gridExtra::grid.arrange(p1, p2, ncol = 2,
                          top = grid::textGrob(
                            "Importancia de variables: MDG vs MDA",
                            gp = grid::gpar(fontsize = 14, fontface = "bold",
                                            col = COLOR_INDIGO)))
} else {
  print(p1)
  print(p2)
}

### 7.1. Comparación de rankings y correlación de Spearman

Calculamos la correlación de Spearman entre los rankings completos para cuantificar el grado de coincidencia entre ambas medidas. Coincidencia alta → conclusión robusta. Discrepancia → posible sesgo de cardinalidad o correlación entre predictoras.

In [ ]:
# Comparación de rankings
ranking_mdg <- order(-imp_df$MDG)
ranking_mda <- order(-imp_df$MDA)

ranking_df <- data.frame(
  variable  = imp_df$variable,
  rank_mdg  = match(seq_along(imp_df$variable), ranking_mdg) - 1,
  rank_mda  = match(seq_along(imp_df$variable), ranking_mda) - 1
)
ranking_df$diff <- ranking_df$rank_mdg - ranking_df$rank_mda

# Correlación de Spearman
rho_spearman <- cor(ranking_df$rank_mdg, ranking_df$rank_mda, method = "spearman")

cat(sprintf("🔍 Correlación de Spearman entre rankings MDG y MDA: %.4f\n", rho_spearman))
if (rho_spearman > 0.85) {
  cat("   (rankings prácticamente equivalentes)\n\n")
} else if (rho_spearman > 0.60) {
  cat("   (rankings consistentes con discrepancias menores)\n\n")
} else {
  cat("   (rankings con discrepancias relevantes)\n\n")
}

cat("🔝 Top 10 por MDG y posición en MDA:\n")
top_mdg <- ranking_df %>% arrange(rank_mdg) %>% head(10)
print(top_mdg)

## 8. Comparativa final: CART, Random Forest y Regresión Logística

Para cerrar la práctica, entrenamos una regresión logística con `glm()` como referencia (modelo de Clase 5 reentrenado rápidamente sobre los mismos datos) y comparamos las tres metodologías sobre las métricas estándar de scoring crediticio.

In [ ]:
# Función auxiliar para calcular KS
calcular_ks <- function(y_true, y_proba) {
  y_num <- as.numeric(as.character(y_true))
  ks_obj <- ks.test(y_proba[y_num == 1], y_proba[y_num == 0])
  as.numeric(ks_obj$statistic)
}

# Función para calcular las tres métricas estándar
calcular_metricas <- function(y_true, y_proba, nombre) {
  auc_val <- as.numeric(auc(y_true, y_proba, quiet = TRUE))
  ks_val  <- calcular_ks(y_true, y_proba)
  gini    <- 2 * auc_val - 1
  data.frame(modelo = nombre, AUC = auc_val, KS = ks_val, Gini = gini)
}

# Entrenamiento rápido de regresión logística como referencia
set.seed(RNG_SEED)
logit <- glm(Y ~ ., data = train, family = binomial(link = "logit"))
p_test_log <- predict(logit, test, type = "response")

# Compilar métricas
metricas_finales <- rbind(
  calcular_metricas(test$Y, p_test_log,  "Reg. Logística"),
  calcular_metricas(test$Y, p_test_cart, "Árbol CART"),
  calcular_metricas(test$Y, p_test_rf,   "Random Forest")
)

cat("📊 Comparativa de modelos sobre German Credit (test set):\n\n")
metricas_finales[, 2:4] <- round(metricas_finales[, 2:4], 4)
print(metricas_finales, row.names = FALSE)

In [ ]:
# Visualización: curvas ROC superpuestas
options(repr.plot.width = 8, repr.plot.height = 8)

# Calcular ROC para los tres modelos
roc_log  <- roc(test$Y, p_test_log,  quiet = TRUE)
roc_cart <- roc(test$Y, p_test_cart, quiet = TRUE)
roc_rf   <- roc(test$Y, p_test_rf,   quiet = TRUE)

roc_df <- bind_rows(
  data.frame(fpr = 1 - roc_log$specificities,  tpr = roc_log$sensitivities,
             modelo = sprintf("Reg. Logística (AUC = %.3f)", as.numeric(roc_log$auc))),
  data.frame(fpr = 1 - roc_cart$specificities, tpr = roc_cart$sensitivities,
             modelo = sprintf("Árbol CART (AUC = %.3f)",     as.numeric(roc_cart$auc))),
  data.frame(fpr = 1 - roc_rf$specificities,   tpr = roc_rf$sensitivities,
             modelo = sprintf("Random Forest (AUC = %.3f)",  as.numeric(roc_rf$auc)))
)

ggplot(roc_df, aes(x = fpr, y = tpr, color = modelo)) +
  geom_line(linewidth = 1.1) +
  geom_abline(intercept = 0, slope = 1, linetype = "dashed",
              color = COLOR_GRAY, alpha = 0.7) +
  scale_color_manual(values = setNames(
    c(COLOR_INDIGO, COLOR_VIOLET, COLOR_AMBER),
    unique(roc_df$modelo)
  )) +
  coord_equal() +
  labs(
    title    = "Curvas ROC — Tres modelos sobre German Credit",
    subtitle = "Test set independiente · partición estratificada 80/20",
    x = "Tasa de falsos positivos (1 - Especificidad)",
    y = "Tasa de verdaderos positivos (Sensibilidad)",
    color = NULL
  ) +
  theme(legend.position = "bottom",
        legend.direction = "vertical")

## 9. Síntesis y entregable

### 📊 Resumen ejecutivo

Cerramos la sesión con la tabla síntesis de los tres modelos y las conclusiones operativas del análisis. La elección final de modelo en un contexto bancario real no se reduce a la métrica de AUC: pesa también la interpretabilidad regulatoria, el costo computacional, la robustez ante deriva temporal y la facilidad de auditoría individual.

In [ ]:
# Tabla síntesis enriquecida
sintesis <- data.frame(
  Criterio = c("AUC-ROC", "KS", "Gini",
               "Interpretabilidad", "Robustez outliers",
               "Manejo categóricas", "Tiempo entrenamiento",
               "Aceptación regulatoria"),
  `Reg. Logística` = c(
    sprintf("%.4f", metricas_finales$AUC[1]),
    sprintf("%.4f", metricas_finales$KS[1]),
    sprintf("%.4f", metricas_finales$Gini[1]),
    "Alta (odds ratios)",
    "Baja",
    "Requiere encoding",
    "Bajo",
    "Estándar de oro"
  ),
  `Árbol CART` = c(
    sprintf("%.4f", metricas_finales$AUC[2]),
    sprintf("%.4f", metricas_finales$KS[2]),
    sprintf("%.4f", metricas_finales$Gini[2]),
    "Muy alta (visual)",
    "Alta",
    "Nativo (factores)",
    "Muy bajo",
    "Aceptable"
  ),
  `Random Forest` = c(
    sprintf("%.4f", metricas_finales$AUC[3]),
    sprintf("%.4f", metricas_finales$KS[3]),
    sprintf("%.4f", metricas_finales$Gini[3]),
    "Media (MDG, MDA)",
    "Muy alta",
    "Nativo (factores)",
    "Medio",
    "Con SHAP/justificación"
  ),
  check.names = FALSE
)

cat("📋 Síntesis comparativa final:\n\n")
print(sintesis, row.names = FALSE, right = FALSE)

### 🔄 Comparación R vs Python

Si compararan este notebook con el análogo en Python ejecutado sobre el mismo dataset y la misma semilla:

| Aspecto | Python (sklearn) | R (rpart + randomForest) |
|---------|------------------|---------------------------|
| **AUC test del CART final** | $\approx 0.73$ | $\approx$ similar |
| **AUC test del Random Forest** | $\approx 0.82$ | $\approx$ similar |
| **Pasos de preprocesamiento** | One-hot manual | No requiere encoding |
| **Selección de poda** | CV externa explícita | `printcp()` y regla 1-SE |
| **Importancia nativa** | Solo MDI | MDG + MDA juntas |
| **Visualización de árbol** | `plot_tree` (texto en cajas) | `rpart.plot` (degradados profesionales) |
| **Tiempo total ejecución** | 5–10 min | 3–5 min (sin overhead OHE) |

> 💡 **R suele ser más rápido para árboles** porque evita el overhead del one-hot encoding y porque `randomForest` está implementado en Fortran. Pero para boosting moderno (XGBoost, LightGBM), Python tiene ventaja por mejor integración con GPU.

### 📝 Lecturas clave del análisis

**1. Performance predictiva**: Random Forest supera consistentemente a CART aislado y, en datasets tabulares de tamaño moderado como el German Credit, suele empatar técnicamente con la regresión logística. La ganancia marginal sobre logística rara vez supera 1–2 puntos de AUC.

**2. Trade-off interpretabilidad-performance**: la regresión logística conserva la transparencia regulatoria mediante coeficientes y odds ratios. CART ofrece reglas auditables linealmente. Random Forest requiere herramientas complementarias (importancia MDA, SHAP) para alcanzar nivel comparable de explicabilidad.

**3. Importancia de variables**: la coincidencia entre MDG y MDA en el ranking de variables top sugiere robustez de las conclusiones. Cuando aparecen discrepancias, conviene investigar correlaciones entre predictoras y posibles sesgos por cardinalidad.

**4. Sobreajuste y poda**: la regla 1-SE de Breiman ofrece una heurística práctica para seleccionar `cp`: elegir el árbol más pequeño cuya performance esté dentro de una desviación estándar del óptimo. Resulta robusta frente a la varianza muestral de la CV interna.

---

### 📋 Entregable para la Clase 7

Sobre la base de este notebook, completen las siguientes consignas para entregar al inicio de la Clase 7 (jueves 21 de mayo):

1. **Replicar el pipeline completo** sobre el German Credit (sin modificaciones).
2. **Reportar las 10 variables más importantes** según MDG y según MDA, presentando ambos rankings lado a lado.
3. **Interpretar económicamente las 3 variables más predictivas**: ¿por qué tiene sentido financiero que sean las top predictoras del default?
4. **Comparar el Random Forest contra la regresión logística** de la Clase 5: ¿la mejora de AUC justificaría el cambio de modelo en producción? ¿qué consideraciones regulatorias agregarían al análisis?
5. **Bonus opcional**: implementar el mismo flujo con el ecosistema moderno `tidymodels` (parsnip + recipes + workflows) y comparar la sintaxis con el flujo clásico.

**Formato**: notebook `.ipynb` (kernel R) subido al campus virtual antes del jueves 21 de mayo, 18:00 hs.

---

### 🌳 Próxima clase — Clase 7 (21 de mayo)

- **Gradient Boosting**: aprendizaje secuencial sobre residuos
- **XGBoost**: regularización, sparsity-aware split, tree pruning
- **SVM**: hiperplano de margen máximo, *kernel trick*
- **Redes neuronales**: MLP, activaciones, backpropagation
- **Calibración**: grid search, random search, optimización bayesiana

> 📚 **Lectura sugerida**: Apunte de Cátedra, Unidad 3, secciones 3.2 y 3.3.  
> Hastie, Tibshirani & Friedman (2009), *The Elements of Statistical Learning*, capítulos 9 y 15.  
> Breiman, L. (2001). *Random Forests*. *Machine Learning*, 45(1), 5–32.
